# AssetFlow — Screen 9 Model Training (Real Team Dataset)
Predictive maintenance (regression), idle asset detection (IsolationForest), retirement scoring, and booking heatmap — trained on the actual team-committed CSVs (`departments`, `categories`, `employees`, `assets`, `allocations`, `maintenance`, `bookings`).

Place the 7 CSVs in a `real_data/` folder next to this notebook before running.

In [1]:
"""
AssetFlow - Model Training Pipeline (REAL team dataset)
Screen 9: Reports & Analytics - ML components

Adapted to the actual schema committed by the team:
    departments, categories, employees, assets, allocations, maintenance, bookings

Models:
    1. Predictive maintenance -> regression on expected maintenance frequency
    2. Idle asset detection -> IsolationForest (multivariate: usage + recency)
    3. Retirement/replacement scoring -> weighted composite score
    4. Booking heatmap -> pure aggregation (no model needed)

Outputs precomputed JSON that frontend/backend consume directly for Screen 9.
"""

import pandas as pd
import numpy as np
import json
import joblib
import os
from datetime import datetime

from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, r2_score

DATA_DIR = "real_data"
OUT_DIR = "real_model_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
departments = pd.read_csv(f"{DATA_DIR}/departments.csv")
categories = pd.read_csv(f"{DATA_DIR}/categories.csv")
employees = pd.read_csv(f"{DATA_DIR}/employees.csv")
assets = pd.read_csv(f"{DATA_DIR}/assets.csv", parse_dates=["acquisition_date"])
allocations = pd.read_csv(
    f"{DATA_DIR}/allocations.csv",
    parse_dates=["allocation_date", "expected_return_date", "actual_return_date"]
)
maintenance = pd.read_csv(
    f"{DATA_DIR}/maintenance.csv", parse_dates=["raised_date", "resolved_date"]
)
bookings = pd.read_csv(f"{DATA_DIR}/bookings.csv", parse_dates=["booking_date"])

# "today" = the most recent activity date across tables, so the pipeline
# stays correct no matter when it's re-run against a refreshed export
TODAY = max(
    assets["acquisition_date"].max(),
    allocations["allocation_date"].max(),
    maintenance["raised_date"].max(),
    bookings["booking_date"].max(),
) + pd.Timedelta(days=1)
print("Using TODAY =", TODAY.date())

# ---------------------------------------------------------------------------
# Feature engineering (one row per asset)
# ---------------------------------------------------------------------------
feat = assets.copy()
feat["age_days"] = (TODAY - feat["acquisition_date"]).dt.days
feat["age_years"] = feat["age_days"] / 365

maint_counts = maintenance.groupby("asset_tag").size().rename("maintenance_count")
last_maint = maintenance.groupby("asset_tag")["raised_date"].max().rename("last_maintenance_date")
feat = feat.merge(maint_counts, on="asset_tag", how="left").merge(last_maint, on="asset_tag", how="left")
feat["maintenance_count"] = feat["maintenance_count"].fillna(0)
feat["days_since_last_maintenance"] = (TODAY - feat["last_maintenance_date"]).dt.days
feat["days_since_last_maintenance"] = feat["days_since_last_maintenance"].fillna(feat["age_days"])

# open/unresolved maintenance issues right now (useful signal, distinct from history count)
open_maint = maintenance[~maintenance["status"].isin(["Resolved", "Rejected"])]
open_counts = open_maint.groupby("asset_tag").size().rename("open_maintenance_count")
feat = feat.merge(open_counts, on="asset_tag", how="left")
feat["open_maintenance_count"] = feat["open_maintenance_count"].fillna(0)

condition_map = {"Excellent": 0, "Good": 1, "Fair": 2, "Poor": 3}
feat["condition_score"] = feat["condition"].map(condition_map)

# usage signal: allocations + bookings in the last 180 days
cutoff_180 = TODAY - pd.Timedelta(days=180)
recent_allocs = allocations[allocations["allocation_date"] >= cutoff_180].groupby("asset_tag").size().rename("recent_allocations")
recent_bookings = bookings[bookings["booking_date"] >= cutoff_180].groupby("asset_tag").size().rename("recent_bookings")
feat = feat.merge(recent_allocs, on="asset_tag", how="left").merge(recent_bookings, on="asset_tag", how="left")
feat["recent_allocations"] = feat["recent_allocations"].fillna(0)
feat["recent_bookings"] = feat["recent_bookings"].fillna(0)
feat["recent_usage_total"] = feat["recent_allocations"] + feat["recent_bookings"]

last_alloc = allocations.groupby("asset_tag")["allocation_date"].max().rename("last_allocation_date")
last_booking = bookings.groupby("asset_tag")["booking_date"].max().rename("last_booking_date")
feat = feat.merge(last_alloc, on="asset_tag", how="left").merge(last_booking, on="asset_tag", how="left")
last_activity = feat[["last_allocation_date", "last_booking_date"]].max(axis=1)
feat["days_since_last_activity"] = (TODAY - last_activity).dt.days
feat["days_since_last_activity"] = feat["days_since_last_activity"].fillna(feat["age_days"])

# overdue-allocation history is a real risk signal in this schema (overdue_flag exists)
overdue_counts = allocations.groupby("asset_tag")["overdue_flag"].sum().rename("overdue_count")
feat = feat.merge(overdue_counts, on="asset_tag", how="left")
feat["overdue_count"] = feat["overdue_count"].fillna(0)

print("Feature table shape:", feat.shape)
print(feat[["asset_tag", "age_years", "maintenance_count", "recent_usage_total",
            "days_since_last_activity", "condition_score", "overdue_count"]].head())

Using TODAY = 2026-07-13
Feature table shape: (350, 25)
  asset_tag  age_years  maintenance_count  recent_usage_total  \
0   AF-0001   2.476712                3.0                 0.0   
1   AF-0002   1.835616                3.0                 0.0   
2   AF-0003   2.652055                2.0                 0.0   
3   AF-0004   0.065753                1.0                 6.0   
4   AF-0005   1.208219                1.0                14.0   

   days_since_last_activity  condition_score  overdue_count  
0                     825.0                1            1.0  
1                     591.0                0            0.0  
2                     880.0                2            2.0  
3                      61.0                1            0.0  
4                       3.0                1            0.0  


## MODEL 1: Predictive maintenance -> regression on maintenance frequency

In [2]:
print("\n--- Model 1: Predictive Maintenance (Regression) ---")

model1_features = ["age_years", "acquisition_cost", "condition_score", "category_id",
                    "recent_usage_total", "overdue_count", "is_bookable"]
X1 = feat[model1_features].copy()
y1 = feat["maintenance_count"]

X1 = pd.get_dummies(X1, columns=["category_id"], prefix="cat")

# With only 350 assets, a single train/test split is noisy (R^2 swung from
# 0.23 to well below that depending on the split). 5-fold CV gives a far
# more honest, stable estimate of real-world performance: ~0.19-0.21 R^2,
# consistent across folds. A shallow tree (max_depth=4) generalizes better
# than a deep one on a dataset this size.
maint_model = RandomForestRegressor(
    n_estimators=300, max_depth=4, min_samples_leaf=5, random_state=42
)
cv_scores = cross_val_score(maint_model, X1, y1, cv=KFold(5, shuffle=True, random_state=42), scoring="r2")
mae_scores = -cross_val_score(
    maint_model, X1, y1, cv=KFold(5, shuffle=True, random_state=42),
    scoring="neg_mean_absolute_error"
)
r2 = cv_scores.mean()
mae = mae_scores.mean()
print(f"5-fold CV  ->  MAE: {mae:.3f}  |  R^2: {r2:.3f} (+/- {cv_scores.std():.3f})")

# fit final model on all data for scoring every asset
maint_model.fit(X1, y1)
feat_importance = pd.Series(maint_model.feature_importances_, index=X1.columns).sort_values(ascending=False)
print("Top feature importances:\n", feat_importance.head())

feat["predicted_maintenance_score"] = maint_model.predict(X1)
mn, mx = feat["predicted_maintenance_score"].min(), feat["predicted_maintenance_score"].max()
feat["maintenance_risk_score"] = ((feat["predicted_maintenance_score"] - mn) / (mx - mn + 1e-9) * 100).round(1)

joblib.dump(maint_model, f"{OUT_DIR}/maintenance_model.joblib")


--- Model 1: Predictive Maintenance (Regression) ---
5-fold CV  ->  MAE: 1.200  |  R^2: 0.200 (+/- 0.099)
Top feature importances:
 age_years             0.457826
acquisition_cost      0.359660
recent_usage_total    0.107249
condition_score       0.029892
cat_C-004             0.014264
dtype: float64


['real_model_outputs/maintenance_model.joblib']

## MODEL 2: Idle asset detection -> IsolationForest (multivariate)

In [3]:
print("\n--- Model 2: Idle Asset Detection (IsolationForest) ---")

idle_features = ["recent_usage_total", "days_since_last_activity"]
X2 = feat[idle_features].copy()

iso = IsolationForest(n_estimators=200, contamination=0.08, random_state=42)
feat["idle_flag_ml"] = iso.fit_predict(X2)
feat["is_idle_ml"] = feat["idle_flag_ml"] == -1

usage_threshold = feat["recent_usage_total"].quantile(0.10)
feat["is_idle_rule"] = feat["recent_usage_total"] <= usage_threshold

print("Idle assets flagged by ML method:", feat["is_idle_ml"].sum())
print("Idle assets flagged by rule method:", feat["is_idle_rule"].sum())

joblib.dump(iso, f"{OUT_DIR}/idle_detector.joblib")


--- Model 2: Idle Asset Detection (IsolationForest) ---
Idle assets flagged by ML method: 28
Idle assets flagged by rule method: 145


['real_model_outputs/idle_detector.joblib']

## MODEL 3: Retirement / replacement scoring (weighted composite, 0-100)

In [4]:
print("\n--- Model 3: Retirement Scoring ---")

def normalize(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

feat["retirement_score"] = (
    0.35 * normalize(feat["age_years"]) +
    0.30 * normalize(feat["maintenance_count"]) +
    0.25 * normalize(feat["condition_score"]) +
    0.10 * normalize(feat["overdue_count"])
) * 100
feat["retirement_score"] = feat["retirement_score"].round(1)
feat["retirement_flag"] = feat["retirement_score"] >= 65

# already-lost/disposed/retired assets shouldn't clutter "retirement candidates"
active_feat = feat[~feat["status"].isin(["Retired", "Disposed", "Lost"])]
print("Assets flagged for retirement review (active only):", active_feat["retirement_flag"].sum())


--- Model 3: Retirement Scoring ---
Assets flagged for retirement review (active only): 4


## MODEL 4: Booking heatmap (pure aggregation, no model)

In [5]:
print("\n--- Booking Heatmap ---")

heatmap = bookings.groupby(["weekday", "start_hour"]).size().reset_index(name="booking_count")
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heatmap["weekday"] = pd.Categorical(heatmap["weekday"], categories=day_order, ordered=True)
heatmap = heatmap.sort_values(["weekday", "start_hour"])


--- Booking Heatmap ---


## Additional plain aggregations

In [6]:
dept_util = (
    allocations.groupby("department_name").size().reset_index(name="allocation_count")
    .sort_values("allocation_count", ascending=False)
)

most_used_assets = (
    pd.concat([
        allocations.groupby("asset_tag").size(),
        bookings.groupby("asset_tag").size()
    ], axis=1).fillna(0).sum(axis=1)
    .rename("usage_count").reset_index()
    .merge(assets[["asset_tag", "asset_name", "category_name"]], on="asset_tag")
    .sort_values("usage_count", ascending=False)
    .head(15)
)

## Export

In [7]:
output = {
    "generated_at": TODAY.strftime("%Y-%m-%d"),
    "predictive_maintenance": {
        "model_metrics": {"mae": round(mae, 3), "r2": round(r2, 3)},
        "feature_importance": feat_importance.round(4).to_dict(),
        "assets": feat[["asset_tag", "asset_name", "category_name",
                         "maintenance_risk_score", "maintenance_count"]]
                  .sort_values("maintenance_risk_score", ascending=False)
                  .head(50).to_dict(orient="records")
    },
    "idle_assets": feat[feat["is_idle_ml"]][
        ["asset_tag", "asset_name", "category_name", "recent_usage_total",
         "days_since_last_activity"]
    ].sort_values("days_since_last_activity", ascending=False).to_dict(orient="records"),
    "retirement_candidates": active_feat[active_feat["retirement_flag"]][
        ["asset_tag", "asset_name", "category_name", "age_years",
         "maintenance_count", "condition", "retirement_score"]
    ].sort_values("retirement_score", ascending=False).to_dict(orient="records"),
    "booking_heatmap": heatmap.to_dict(orient="records"),
    "department_utilization": dept_util.to_dict(orient="records"),
    "most_used_assets": most_used_assets[["asset_tag", "asset_name", "category_name", "usage_count"]]
                        .to_dict(orient="records")
}

with open(f"{OUT_DIR}/screen9_analytics.json", "w") as f:
    json.dump(output, f, indent=2, default=str)

print(f"\nAll outputs saved to {OUT_DIR}/screen9_analytics.json")
print("Models saved: maintenance_model.joblib, idle_detector.joblib")


All outputs saved to real_model_outputs/screen9_analytics.json
Models saved: maintenance_model.joblib, idle_detector.joblib
